In [10]:
import pandas as pd
import folium
import json
import numpy as np
import branca.colormap as cm

# -------------------------
# 1 Load branche data
# -------------------------

df = pd.read_excel("Branche.xlsx")


# Kommune columns start after the first 3 columns
branche_data = df.iloc[:,3:]

# Sum passengers per airport
branche_totals = branche_data.sum()

# -------------------------
# 2 Airport → municipality
# -------------------------

branche_to_muni = {
    "København": "Københavns Kommune",
    "Frederiksberg": "Frederiksberg Kommune",
    "Dragør": "Dragør Kommune",
    "Tårnby": "Tårnby Kommune",
    "Albertslund": "Albertslund Kommune",
    "Ballerup": "Ballerup Kommune",
    "Brøndby": "Brøndby Kommune",
    "Gentofte": "Gentofte Kommune",
    "Gladsaxe": "Gladsaxe Kommune",
    "Glostrup": "Glostrup Kommune",
    "Herlev": "Herlev Kommune",
    "Hvidovre": "Hvidovre Kommune",
    "Høje-Taastrup": "Høje-Taastrup Kommune",
    "Ishøj": "Ishøj Kommune",
    "Lyngby-Taarbæk": "Lyngby-Taarbæk Kommune",
    "Rødovre": "Rødovre Kommune",
    "Vallensbæk": "Vallensbæk Kommune",
    "Allerød": "Allerød Kommune",
    "Egedal": "Egedal Kommune",
    "Fredensborg": "Fredensborg Kommune",
    "Frederikssund": "Frederikssund Kommune",
    "Furesø": "Furesø Kommune",
    "Gribskov": "Gribskov Kommune",
    "Halsnæs": "Halsnæs Kommune",
    "Helsingør": "Helsingør Kommune",
    "Hillerød": "Hillerød Kommune",
    "Hørsholm": "Hørsholm Kommune",
    "Rudersdal": "Rudersdal Kommune",
    "Bornholm": "Bornholms Regionskommune",

    "Greve": "Greve Kommune",
    "Køge": "Køge Kommune",
    "Lejre": "Lejre Kommune",
    "Roskilde": "Roskilde Kommune",
    "Solrød": "Solrød Kommune",
    "Faxe": "Faxe Kommune",
    "Guldborgsund": "Guldborgsund Kommune",
    "Holbæk": "Holbæk Kommune",
    "Kalundborg": "Kalundborg Kommune",
    "Lolland": "Lolland Kommune",
    "Næstved": "Næstved Kommune",
    "Odsherred": "Odsherred Kommune",
    "Ringsted": "Ringsted Kommune",
    "Slagelse": "Slagelse Kommune",
    "Sorø": "Sorø Kommune",
    "Stevns": "Stevns Kommune",
    "Vordingborg": "Vordingborg Kommune",

    "Assens": "Assens Kommune",
    "Faaborg-Midtfyn": "Faaborg-Midtfyn Kommune",
    "Kerteminde": "Kerteminde Kommune",
    "Langeland": "Langeland Kommune",
    "Middelfart": "Middelfart Kommune",
    "Nordfyns": "Nordfyns Kommune",
    "Nyborg": "Nyborg Kommune",
    "Odense": "Odense Kommune",
    "Svendborg": "Svendborg Kommune",
    "Ærø": "Ærø Kommune",

    "Billund": "Billund Kommune",
    "Esbjerg": "Esbjerg Kommune",
    "Fanø": "Fanø Kommune",
    "Fredericia": "Fredericia Kommune",
    "Haderslev": "Haderslev Kommune",
    "Kolding": "Kolding Kommune",
    "Sønderborg": "Sønderborg Kommune",
    "Tønder": "Tønder Kommune",
    "Varde": "Varde Kommune",
    "Vejen": "Vejen Kommune",
    "Vejle": "Vejle Kommune",
    "Aabenraa": "Aabenraa Kommune",

    "Favrskov": "Favrskov Kommune",
    "Hedensted": "Hedensted Kommune",
    "Horsens": "Horsens Kommune",
    "Norddjurs": "Norddjurs Kommune",
    "Odder": "Odder Kommune",
    "Randers": "Randers Kommune",
    "Samsø": "Samsø Kommune",
    "Silkeborg": "Silkeborg Kommune",
    "Skanderborg": "Skanderborg Kommune",
    "Syddjurs": "Syddjurs Kommune",
    "Aarhus": "Aarhus Kommune",

    "Herning": "Herning Kommune",
    "Holstebro": "Holstebro Kommune",
    "Ikast-Brande": "Ikast-Brande Kommune",
    "Lemvig": "Lemvig Kommune",
    "Ringkøbing-Skjern": "Ringkøbing-Skjern Kommune",
    "Skive": "Skive Kommune",
    "Struer": "Struer Kommune",
    "Viborg": "Viborg Kommune",

    "Brønderslev": "Brønderslev Kommune",
    "Frederikshavn": "Frederikshavn Kommune",
    "Hjørring": "Hjørring Kommune",
    "Jammerbugt": "Jammerbugt Kommune",
    "Læsø": "Læsø Kommune",
    "Mariagerfjord": "Mariagerfjord Kommune",
    "Morsø": "Morsø Kommune",
    "Rebild": "Rebild Kommune",
    "Thisted": "Thisted Kommune",
    "Vesthimmerlands": "Vesthimmerlands Kommune",
    "Aalborg": "Aalborg Kommune"
}

muni_values = {}
muni_airport = {}

for airport, value in airport_totals.items():

    muni = airport_to_muni.get(airport)

    if muni is not None:
        muni_values[muni] = float(value)
        muni_airport[muni] = airport

# -------------------------
# 3 Load GeoJSON
# -------------------------

with open("kommune.geojson") as f:
    kommuner = json.load(f)

features = [
    f for f in kommuner["features"]
    if f["geometry"]["type"] in ["Polygon", "MultiPolygon"]
]

# -------------------------
# 4 Attach airport + passengers
# -------------------------

for feature in features:

    name = feature["properties"]["name"]

    value = muni_values.get(name, 0)
    airport = muni_airport.get(name, "None")

    feature["properties"]["airport"] = airport
    feature["properties"]["passengers"] = f"{int(value):,}"

# -------------------------
# 5 Create map
# -------------------------

m = folium.Map(location=[56.2, 10.0], zoom_start=7)

# -------------------------
# 6 Log-scaled color scale
# -------------------------

log_values = {k: np.log1p(v) for k, v in muni_values.items()}

min_log = min(log_values.values())
max_log = max(log_values.values())

colormap = cm.linear.OrRd_09.scale(min_log, max_log)

# -------------------------
# 7 Style municipalities
# -------------------------

def style_function(feature):

    name = feature["properties"]["name"]

    value = muni_values.get(name)

    if value is None:
        return {
            "fillOpacity": 0,
            "color": "black",
            "weight": 1
        }

    return {
        "fillColor": colormap(np.log1p(value)),
        "color": "black",
        "weight": 1,
        "fillOpacity": 0.9
    }

# -------------------------
# 8 Draw municipalities
# -------------------------

folium.GeoJson(
    {"type": "FeatureCollection", "features": features},
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(
        fields=["name", "airport", "passengers"],
        aliases=["Municipality:", "Airport:", "Passengers:"],
        localize=True
    )
).add_to(m)

# -------------------------
# 9 Legend
# -------------------------

colormap.caption = "Domestic passengers ≥2020 (log scale)"
colormap.add_to(m)

# -------------------------
# 10 Save map
# -------------------------

m.save("domestic_airfields_heatmap.html")

print("Map saved as domestic_airfields_heatmap.html")

Map saved as domestic_airfields_heatmap.html
